# JPEG self-similarity experiment
Compare jpeg_similarity_control and jpeg_similarity on the same server, seed and batch.
IIS-inspired JPEG ablation, without GFNet/Laplacian.
See configs/experiments/README.md. The last cell starts training.


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if not (ROOT / 'src').is_dir():
    raise RuntimeError('Open this notebook from the project root or notebooks directory')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import os
os.chdir(ROOT)
import numpy as np  # Initialize NumPy before PyTorch on Windows.

from src.config import load_experiment_config

experiment = 'jpeg_similarity'  # or jpeg_similarity_control
cfg = load_experiment_config(ROOT / 'configs' / 'experiments' / f'{experiment}.yaml')
cfg


In [ ]:
import torch
from src.training.builders import build_model
from src.budget import count_gflops
from src.eval.protocol import EvaluationProtocol

protocol = EvaluationProtocol.load(cfg.dataset.protocol_path)
print('Train/development:', len(protocol.rows('train')), len(protocol.rows('development')))
native_size = (1024, 1024)
with torch.device('meta'):
    budget_model = build_model(cfg.model, aux_weight=cfg.loss.aux_weight, pretrained=False).eval()
    gflops = count_gflops(budget_model, cfg.dataset.image_size,
                          native_size=native_size)
del budget_model
assert gflops <= 100, f'{gflops:.2f} GFLOPs exceeds 100'
print(f'Full inference: {gflops:.3f} GFLOPs')
if native_size is not None:
    print('Reference native JPEG size:', native_size, '; larger sources may exceed 100 GFLOPs')


In [ ]:
from src.training.engine import run_experiment

run = run_experiment(cfg)
run.summary
